In [2]:
import kagglehub
path = kagglehub.dataset_download("omkargurav/face-mask-dataset")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'face-mask-dataset' dataset.
Path to dataset files: /kaggle/input/face-mask-dataset


In [3]:
import shutil
shutil.copytree(path, "data")

'data'

In [22]:
!pip install opencv-python-headless tensorflow scikit-learn

In [4]:
import os
import cv2
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model
from google.colab.patches import cv2_imshow

In [5]:
IMG_SIZE = 64
data = []
labels = []

categories = ["with_mask", "without_mask"]

for label, category in enumerate(categories):
    path = os.path.join("/content/data/data", category)

    for img in os.listdir(path):
        img_path = os.path.join(path, img)

        image = cv2.imread(img_path)
        if image is None:
            continue

        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        data.append(image)
        labels.append(label)

data = np.array(data, dtype="float32") / 255.0
labels = to_categorical(labels, 2)

X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42, shuffle=True
)

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(64, activation='relu'),
    Dense(2, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

model.save("mask_model.h5")

print("✅ Model trained and saved successfully!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
189/189 ━━━━━━━━━━━━━━━━━━━━ 34s 168ms/step - accuracy: 0.8347 - loss: 0.3802 - val_accuracy: 0.9014 - val_loss: 0.2486
Epoch 2/5
189/189 ━━━━━━━━━━━━━━━━━━━━ 39s 159ms/step - accuracy: 0.9067 - loss: 0.2306 - val_accuracy: 0.9140 - val_loss: 0.2166
Epoch 3/5
189/189 ━━━━━━━━━━━━━━━━━━━━ 30s 161ms/step - accuracy: 0.9307 - loss: 0.1830 - val_accuracy: 0.9246 - val_loss: 0.1895
Epoch 4/5
189/189 ━━━━━━━━━━━━━━━━━━━━ 40s 155ms/step - accuracy: 0.9441 - loss: 0.1474 - val_accuracy: 0.9226 - val_loss: 0.2057
Epoch 5/5
189/189 ━━━━━━━━━━━━━━━━━━━━ 41s 158ms/step - accuracy: 0.9603 - loss: 0.1112 - val_accuracy: 0.9292 - val_loss: 0.1919


✅ Model trained and saved successfully!


In [9]:
model = load_model("mask_model.h5")

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]

        # ✅ Resize (match training size!)
        face = cv2.resize(face, (64,64))

        # ✅ Convert BGR → RGB
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

        # ✅ Normalize
        face = face / 255.0

        # ✅ Reshape
        face = np.reshape(face, (1,64,64,3))

        pred = model.predict(face)

        # ✅ Binary classification fix
        if pred[0][0] > 0.5:
            text = "No Mask"
            color = (0,0,255)
        else:
            text = "Mask"
            color = (0,255,0)

        cv2.rectangle(frame, (x,y), (x+w,y+h), color, 2)
        cv2.putText(frame, text, (x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("Mask Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [13]:
from google.colab import files
uploaded = files.upload()

Saving 40199740_1-care-4-all-face-mask-5-layer-in-multicolor-without-valve.webp to 40199740_1-care-4-all-face-mask-5-layer-in-multicolor-without-valve (2).webp


In [16]:
# Load model
model = load_model("mask_model.h5")

# Load image
img_path = "your_image.jpg"   # change this
img = cv2.imread(img_path)

# Preprocess
img_resized = cv2.resize(img, (64,64))
img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
img_norm = img_rgb / 255.0
img_input = np.reshape(img_norm, (1,64,64,3))

# Predict
pred = model.predict(img_input)
print("Prediction:", pred)

# Softmax handling
label = np.argmax(pred)

if label == 0:
    print("❌ No Mask Detected")
else:
    print("😷 Mask Detected")

error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


In [ ]:
model = load_model("mask_model.h5")

In [ ]:
print(model.summary())
print(model.output_shape)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       802,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 822,404 (3.14 MB)

 Trainable params: 822,402 (3.14 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

None
(None, 2)


In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from base64 import b64decode

In [17]:
def take_photo():

    js = Javascript("""
        async function takePhoto() {

            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            const video = document.createElement('video');
            video.style.display = 'block';
            video.srcObject = stream;

            document.body.appendChild(video);
            await video.play();

            // wait for camera warm-up
            await new Promise(resolve => setTimeout(resolve, 1500));

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;

            canvas.getContext('2d').drawImage(video, 0, 0);

            // 🔥 HARD STOP CAMERA
            stream.getTracks().forEach(track => {
                track.stop();
            });

            video.pause();
            video.srcObject = null;
            video.remove();

            return canvas.toDataURL('image/jpeg', 0.8);
        }
    """)

    display(js)

    data = eval_js("takePhoto()")

    binary = b64decode(data.split(',')[1])
    img = np.frombuffer(binary, np.uint8)
    img = cv2.imdecode(img, cv2.IMREAD_COLOR)

    return img

In [18]:
faces = face_cascade.detectMultiScale(
    gray,
    scaleFactor=1.1,
    minNeighbors=3,
    minSize=(60, 60)
)

NameError: name 'gray' is not defined

In [19]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

# STEP 1: Take photo
frame = take_photo()

# STEP 2: Face detection
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
faces = face_cascade.detectMultiScale(gray, 1.3, 5)

# STEP 3: Check face
if len(faces) == 0:
    print("No face detected")
    cv2_imshow(frame)

else:
    for (x, y, fw, fh) in faces:

        # STEP 4: crop face properly
        pad = 30
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(frame.shape[1], x + fw + pad)
        y2 = min(frame.shape[0], y + fh + pad)

        face = frame[y1:y2, x1:x2]

        # STEP 5: FIX INPUT SIZE (VERY IMPORTANT)
        h, w = model.input_shape[1], model.input_shape[2]

        img = cv2.resize(face, (w, h))
        img = img.astype("float32") / 255.0
        img = np.expand_dims(img, axis=0)

        # STEP 6: PREDICTION
        pred = model.predict(img)[0]

        print("Raw prediction:", pred)  # DEBUG

        mask_prob = pred[0]
        no_mask_prob = pred[1]

        # STEP 7: DECISION
        if mask_prob > no_mask_prob:
            label = "Mask"
            confidence = mask_prob
            color = (0, 255, 0)
        else:
            label = "No Mask"
            confidence = no_mask_prob
            color = (0, 0, 255)

        print("Final:", label, confidence)

        # STEP 8: DRAW RESULT
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame,
                    f"{label} {confidence:.2f}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2)

    # STEP 9: SHOW FINAL OUTPUT
    cv2_imshow(frame)

NameError: name 'Javascript' is not defined

In [20]:
print(model.predict(img))

KeyError: 'pop from an empty set'

In [21]:
print(pred)
print(model.output_shape)

[[0.9541095  0.04589052]]
(None, 2)
